# 4q QCNN — Noiseless multiseed (local Mac run)

Serial R=10 seed loop for the noiseless branch of the Wave-K cross-architecture comparison.
Identical architecture to the noisy run on the cluster (`run_4q.py` + `parallel_quanv_sim.py`),
solo cambia il `backend` da `sim_noisy` a `sim_noiseless`.

**Output**: `results_4q_noiseless/results_seed_<NNN>.json` per ognuno dei 10 seed.
Same structure as the noisy run, so `wilcoxon_cross_arch.py` can run the paired test.

**Pre-requisiti**:
- `parallel_quanv_sim.py` and `run_4q.py` in the same directory (or in `sys.path`)
- EuroSAT dataset in `./dataset/training` and `./dataset/validation` (same as the 9q run)
- Pacchetti: `qiskit==2.2`, `qiskit-aer>=0.15`, `torch`, `torchvision`, `pillow`, `numpy`

**Tempo atteso**: ~5–10 min totali su laptop (i9 o M-series). Statevector 4q è ridicolmente veloce.

## 1 — Env vars + imports

**IMPORTANT**: the BLAS env vars must be set BEFORE importing numpy/torch.

In [ ]:
import os
# Conservativo, va bene su laptop. Su Mac M-series puoi alzare OMP a 4-8.
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

import sys, json, time
from pathlib import Path
import numpy as np

# Aggiungi la dir corrente al path (se hai messo run_4q.py altrove, edita qui)
sys.path.insert(0, '.')

from run_4q import Config4q, train_one_seed

print('Imports OK')

## 2 — Noiseless config

Identical to the noisy run on the cluster EXCEPT for `backend_type='sim_noiseless'`.
Paired with the seeds of the 9q-noiseless run (multiseed `base_seed=42`, R=10).

In [ ]:
# ── Edita questi path secondo la tua macchina ──
TRAIN_DIR = './dataset/training'
VAL_DIR   = './dataset/validation'
OUTPUT_DIR = './results_4q_noiseless'

# ── Seed (pairing con multiseed 9q) ──
BASE_SEED = 42
R = 10
SEEDS = list(range(BASE_SEED, BASE_SEED + R))   # [42, 43, ..., 51]
print(f'Seeds: {SEEDS}')

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 3 — Loop R=10 seeds (serial)

To resume mid-way (e.g. seeds already done), just relaunch:
the skip is automatic via a check on the output JSON file.

In [ ]:
t_global = time.time()
all_results = []

for seed in SEEDS:
    out_path = Path(OUTPUT_DIR) / f'results_seed_{seed:03d}.json'
    if out_path.exists():
        print(f'\n[seed {seed}] SKIP (already done: {out_path})')
        with open(out_path) as f:
            all_results.append(json.load(f))
        continue

    print(f'\n{"="*70}')
    print(f'  Seed {seed}/{SEEDS[-1]}   (run {SEEDS.index(seed)+1} of {len(SEEDS)})')
    print(f'{"="*70}')

    cfg = Config4q(
        seed=seed,
        backend_type='sim_noiseless',
        train_dir=TRAIN_DIR,
        val_dir=VAL_DIR,
        output_dir=OUTPUT_DIR,
        aer_max_parallel_experiments=4,   # 4 workers are enough on a laptop
        # Everything else stays at default (= identical to the noisy run)
    )
    result = train_one_seed(cfg, verbose=True)
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  ✓ Saved: {out_path}')
    all_results.append(result)

wall_total = time.time() - t_global
print(f'\n{"="*70}')
print(f'  ALL DONE in {wall_total/60:.1f} min')
print(f'{"="*70}')


## 4 — Per-seed metrics summary

In [ ]:
print(f'{"seed":>6} {"final_val_acc":>15} {"best_val_acc":>15} {"wall (min)":>12}')
print('-'*55)
for r in all_results:
    print(f'{r["seed"]:>6} {r["final_val_acc"]:>15.4f} {r["best_val_acc"]:>15.4f} '
          f'{r["wall_time_s"]/60:>12.1f}')

final_accs = [r['final_val_acc'] for r in all_results]
print('-'*55)
print(f'  Mean ± std final_val_acc: {np.mean(final_accs):.4f} ± {np.std(final_accs):.4f}')
print(f'  Min / max:                {np.min(final_accs):.4f} / {np.max(final_accs):.4f}')

## 5 — (Optional) Consolidated aggregate

Saves a single summary file with all 10 seeds. Useful for `project_4q_to_9q.py` and `wilcoxon_cross_arch.py`.

In [ ]:
summary = {
    'architecture': '4q_noiseless_aer_statevector',
    'n_seeds': len(all_results),
    'seeds': [r['seed'] for r in all_results],
    'final_val_acc': [r['final_val_acc'] for r in all_results],
    'best_val_acc': [r['best_val_acc'] for r in all_results],
    'mean_final_val_acc': float(np.mean([r['final_val_acc'] for r in all_results])),
    'std_final_val_acc':  float(np.std([r['final_val_acc'] for r in all_results])),
}
summary_path = Path(OUTPUT_DIR) / 'summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'  ✓ Summary saved: {summary_path}')
summary